In [2]:
!pip install -q pymupdf langchain-text-splitters tiktoken sentence-transformers faiss-cpu google-genai

In [4]:
from google.colab import files

uploaded = files.upload()

Saving intro-to-ml-book.pdf to intro-to-ml-book.pdf


In [5]:
import re
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# 1. Load the PDF document
pdf_path = 'intro-to-ml-book.pdf'  # Matches your uploaded file
doc = fitz.open(pdf_path)

# 2. Extract text from all pages
pages_data = [
    {'page_number': i + 1, 'text': doc[i].get_text('text')}
    for i in range(len(doc))
]

# 3. Explore document structure
total_pages = len(doc)
empty_pages = [p['page_number'] for p in pages_data if not p['text'].strip()]
short_pages = [
    p['page_number'] for p in pages_data if 0 < len(p['text'].strip()) < 100
]

print('=== PART 1: DOCUMENT EXPLORATION ===')
print(f'Total Pages: {total_pages}')
print(
    f'Empty Pages ({len(empty_pages)}):'
    f' {empty_pages[:10]}'
)  # Covers/blank pages
print(
    f'Low-Text Pages (<100 chars) ({len(short_pages)}):'
    f' {short_pages[:10]}'
)  # Chapter dividers/diagrams


# Clean common PDF text quality issues (broken hyphenation & excessive whitespace)
def clean_pdf_text(text):
  text = re.sub(
      r'(\w+)-\n(\w+)', r'\1\2', text
  )  # Fix split words like "regres-\nsion"
  text = re.sub(r'\n{3,}', '\n\n', text)  # Collapse extra blank lines
  return text.strip()


for p in pages_data:
  p['clean_text'] = clean_pdf_text(p['text'])

# 4. Split text into token-based chunks (500-1000 tokens, 50-150 overlap)
tokenizer = tiktoken.get_encoding('cl100k_base')
count_tokens = lambda text: len(tokenizer.encode(text))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,  # Within 500–1000 tokens
    chunk_overlap=100,  # Within 50–150 tokens
    length_function=count_tokens,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

chunks = []
for p in pages_data:
  if not p['clean_text']:
    continue
  for idx, chunk_text in enumerate(text_splitter.split_text(p['clean_text'])):
    chunks.append({
        'chunk_id': f"p{p['page_number']}_c{idx+1}",
        'page_number': p['page_number'],
        'token_count': count_tokens(chunk_text),
        'text': chunk_text,
    })

print(f'Total Chunks Created: {len(chunks)}')
print(
    'Average Tokens per Chunk:'
    f" {sum(c['token_count'] for c in chunks) / len(chunks):.1f}"
)

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


=== PART 1: DOCUMENT EXPLORATION ===
Total Pages: 392
Empty Pages (5): [2, 144, 224, 318, 336]
Low-Text Pages (<100 chars) (0): []
Total Chunks Created: 402
Average Tokens per Chunk: 424.2


In [6]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print('=== PART 2: EMBEDDING & VECTOR DATABASE ===')
# 1. Load SentenceTransformer embedding model (384-dimensional dense vectors)
embed_model_name = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(embed_model_name)

# Generate normalized embeddings for cosine similarity
texts = [c['text'] for c in chunks]
embeddings = embedder.encode(
    texts, show_progress_bar=True, normalize_embeddings=True
)
embeddings = np.array(embeddings).astype('float32')

# 2. Store embeddings in FAISS (IndexFlatIP performs exact Cosine Similarity on normalized vectors)
dimension = embeddings.shape[1]
vector_index = faiss.IndexFlatIP(dimension)
vector_index.add(embeddings)

print(
    f'Stored {vector_index.ntotal} vectors of dimension {dimension} in FAISS.'
)


# 3. Implement Similarity Search / Top-k Retrieval Pipeline (Part 3)
def retrieve_chunks(query, k=5):
  query_vec = embedder.encode([query], normalize_embeddings=True).astype(
      'float32'
  )
  scores, indices = vector_index.search(query_vec, k)

  results = []
  for score, idx in zip(scores[0], indices[0]):
    if idx != -1:
      results.append({
          'score': float(score),
          'chunk_id': chunks[idx]['chunk_id'],
          'page_number': chunks[idx]['page_number'],
          'text': chunks[idx]['text'],
      })
  return results


# Deliverable: Example of retrieved chunks & Experimenting with k = 3, 5, 7
sample_query = 'What is the difference between overfitting and underfitting?'
print(f'\n=== PART 3: RETRIEVAL EXPERIMENT (Query: "{sample_query}") ===')

for k_val in [3, 5, 7]:
  retrieved = retrieve_chunks(sample_query, k=k_val)
  pages_found = [r['page_number'] for r in retrieved]
  top_score = retrieved[0]['score']
  print(
      f'k={k_val} -> Top Similarity Score: {top_score:.4f} | Retrieved Pages:'
      f' {pages_found}'
  )

print('\n--- Top Retrieved Chunk Preview (k=3, Rank 1) ---')
top_hit = retrieve_chunks(sample_query, k=3)[0]
print(
    f"[Page {top_hit['page_number']} | Score: {top_hit['score']:.4f}]\n{top_hit['text'][:450]}..."
)

=== PART 2: EMBEDDING & VECTOR DATABASE ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Stored 402 vectors of dimension 384 in FAISS.

=== PART 3: RETRIEVAL EXPERIMENT (Query: "What is the difference between overfitting and underfitting?") ===
k=3 -> Top Similarity Score: 0.3849 | Retrieved Pages: [137, 73, 6]
k=5 -> Top Similarity Score: 0.3849 | Retrieved Pages: [137, 73, 6, 386, 43]
k=7 -> Top Similarity Score: 0.3849 | Retrieved Pages: [137, 73, 6, 386, 43, 61, 88]

--- Top Retrieved Chunk Preview (k=3, Rank 1) ---
[Page 137 | Score: 0.3849]
13 Because the probabilities are floating-point numbers, it is unlikely that they will both be exactly 0.500. How‐
ever, if that happens, the prediction is made at random.
Out[113]:
Predicted probabilities:
[[ 0.016  0.984]
 [ 0.843  0.157]
 [ 0.981  0.019]
 [ 0.974  0.026]
 [ 0.014  0.986]
 [ 0.025  0.975]]
Because the probabilities for the two classes sum to 1, exactly one of the classes will
be above 50% certainty. That class is the one that i...


In [11]:
import os
from google import genai
from google.genai import types

# Set your Gemini API Key
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"
client = genai.Client()


def build_grounded_prompt(query, retrieved_chunks):
  context_blocks = []
  for i, chunk in enumerate(retrieved_chunks, 1):
    context_blocks.append(
        f"[Source {i} - Page {chunk['page_number']}]:\n{chunk['text']}"
    )
  context_str = "\n\n---\n\n".join(context_blocks)

  system_instruction = (
      "You are a strict Machine Learning Knowledge Assistant. "
      "Answer the user's question using ONLY the provided book excerpts. "
      "Cite the page numbers in parentheses like (Page X) for every claim. "
      "If the provided context does not contain the answer, explicitly state: "
      "'I cannot find the answer to this question in the provided textbook excerpts.' "
      "Do NOT use outside knowledge or fabricate facts."
  )

  user_prompt = f"TEXTBOOK CONTEXT:\n{context_str}\n\nQUESTION: {query}"
  return system_instruction, user_prompt


def ask_ml_assistant(query, k=5):
  # Step A: Retrieve top-k relevant chunks
  retrieved = retrieve_chunks(query, k=k)

  # Step B: Construct grounded prompt
  sys_prompt, user_prompt = build_grounded_prompt(query, retrieved)

  # Step C: Generate response with low temperature (0.1) to prevent hallucination
  response = client.models.generate_content(
      model="gemini-3.5-flash-lite",
      contents=user_prompt,
      config=types.GenerateContentConfig(
          system_instruction=sys_prompt,
          temperature=0.1,
      ),
  )

  return {
      "query": query,
      "answer": response.text,
      "sources": [(r["page_number"], round(r["score"], 4)) for r in retrieved],
  }


# === PART 4 & 5: DEMONSTRATION OF MULTIPLE QUERIES ===
test_queries = [
    "How does k-Nearest Neighbors (k-NN) make predictions for classification?",
    "What is cross-validation and why do we use it instead of a single train-test split?",
    "What is the recipe for baking a chocolate cake?",  # Hallucination / out-of-domain test
]

print("=== PART 4 & 5: END-TO-END RAG DEMONSTRATION ===")
for q in test_queries:
  result = ask_ml_assistant(q, k=5)
  print(f"\nQ: {result['query']}")
  print(f"Retrieved (Page, Similarity): {result['sources']}")
  print(f"A:\n{result['answer']}")
  print("-" * 70)

# Interactive Real-Time Q&A Loop (Type 'exit' to stop)
while True:
  user_q = input("\nAsk a Machine Learning question (or type 'exit'): ")
  if user_q.strip().lower() == "exit":
    break
  if not user_q.strip():
    continue
  res = ask_ml_assistant(user_q, k=5)
  print(f"\nAnswer:\n{res['answer']}")
  print(f"Sources Cited (Page, Score): {res['sources']}")

=== PART 4 & 5: END-TO-END RAG DEMONSTRATION ===

Q: How does k-Nearest Neighbors (k-NN) make predictions for classification?
Retrieved (Page, Similarity): [(49, 0.7699), (35, 0.7607), (54, 0.6937), (50, 0.6693), (38, 0.6518)]
A:
Based on the provided textbook excerpts, the k-Nearest Neighbors (k-NN) algorithm makes predictions for classification in the following ways:

* **In its simplest version (using a single neighbor):** The algorithm considers exactly one nearest neighbor, which is the closest training data point to the point you want to make a prediction for. The prediction is simply the known output (or label) for that training point (Page 49).
* **When considering multiple neighbors ($k$ neighbors):** Instead of using only the closest neighbor, the algorithm can consider any fixed number $k$ of neighbors in the training set (such as the closest three or five neighbors) (Pages 35, 50). When considering more than one neighbor, it uses voting by counting how many neighbors belong